In [1]:
# Learning Pytorch
#      - Try out some pytorch tensor methods, e.g. torch.randn, torch.tensor.unfold, tensor_split.
# 
# Learning Pytroch Lightning data preparing
#      - Try out **Sampler methods to (random) sort and batch datasets
#      - Use Dataloader method with a new created sub class of Dataset to load/iterate datasets, 
#        which is feed in the same way to the method training_step defined by the pytorch Lightning 
#      - Try out the methods pad_sequence and pack_padded_sequence ect. 
#        These methods are required by rnn and derivatives nets like LSTM to keep the train datasets having same length.

In [2]:
import torch
import numpy

In [3]:
# create 3 rows 5 colums tensor data
x = torch.randn(3,5)
# transform it to 5 rows 3 colums
x = x.t()
print(x)
print(type(x))

tensor([[ 1.9787,  0.6051, -0.8238],
        [ 0.0052,  0.6074, -0.7175],
        [-0.9696,  0.6189, -1.0195],
        [ 0.4197, -0.3087, -0.3659],
        [-0.7681, -0.0949,  1.0889]])
<class 'torch.Tensor'>


In [4]:
# unfold(dimension, size, step) 
window_length = 2
# along the 1st dimension(value 0) take 2(window_length), go to the 2th(step) element repeat, the last will be dropped
print(x.unfold(0, 2, 2)) 
# along the 2nd dimension(value 1) take 2(window_length) , go to the 9th(step) element repeat
print(x.unfold(1, 2, 5)) 

tensor([[[ 1.9787,  0.0052],
         [ 0.6051,  0.6074],
         [-0.8238, -0.7175]],

        [[-0.9696,  0.4197],
         [ 0.6189, -0.3087],
         [-1.0195, -0.3659]]])
tensor([[[ 1.9787,  0.6051]],

        [[ 0.0052,  0.6074]],

        [[-0.9696,  0.6189]],

        [[ 0.4197, -0.3087]],

        [[-0.7681, -0.0949]]])


In [5]:
x = torch.arange(9)
print(torch.tensor_split(x, torch.tensor([1,4,6])))

x = torch.arange(7)
print('split 0 ... 6 in 3, 2 ,2 ')
# If indices_or_sections is an integer n or a zero dimensional long tensor with value n, 
# input is split into n sections along dimension dim. If input is divisible by n along dimension dim, 
# each section will be of equal size, input.size(dim) / n. If input is not divisible by n, 
# the sizes of the first int(input.size(dim) % n) sections will have size int(input.size(dim) / n) + 1, 
# and the rest will have size int(input.size(dim) / n).
print(torch.tensor_split(x, 3))
print(torch.tensor_split(x, (1, 6)))

x = torch.arange(14).reshape(2, 7)
print(torch.tensor_split(x, 3, dim=1))


x = torch.arange(14).reshape(7, 2)
print(x)
print(torch.tensor_split(x, 3, dim=0))

#print(x,torch.tensor_split(x, (2, 6), dim=0))

(tensor([0]), tensor([1, 2, 3]), tensor([4, 5]), tensor([6, 7, 8]))
split 0 ... 6 in 3, 2 ,2 
(tensor([0, 1, 2]), tensor([3, 4]), tensor([5, 6]))
(tensor([0]), tensor([1, 2, 3, 4, 5]), tensor([6]))
(tensor([[0, 1, 2],
        [7, 8, 9]]), tensor([[ 3,  4],
        [10, 11]]), tensor([[ 5,  6],
        [12, 13]]))
tensor([[ 0,  1],
        [ 2,  3],
        [ 4,  5],
        [ 6,  7],
        [ 8,  9],
        [10, 11],
        [12, 13]])
(tensor([[0, 1],
        [2, 3],
        [4, 5]]), tensor([[6, 7],
        [8, 9]]), tensor([[10, 11],
        [12, 13]]))


In [6]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

df = pd.DataFrame({
    'feature1': range(10),
    'feature2': range(10,20),
    'feature3': range(20,30)
})

# Convert the DataFrame to a PyTorch tensor
tensor_data = torch.tensor(df.values, dtype=torch.float32)

# Wrap the tensor in a TensorDataset
dataset = TensorDataset(tensor_data)
print(dataset)

# Create a DataLoader to split the data into batches
batch_size = 3  # 20 samples will give 6 full batches and 1 with 2 samples
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

# Step 5: Iterate through the batches and print them
for i, batch in enumerate(dataloader):
    print(f"Batch {i+1}:")
    print(batch[0])  # batch is a tuple; we extract the tensor

Batch 1:
tensor([[ 0., 10., 20.],
        [ 1., 11., 21.],
        [ 2., 12., 22.]])
Batch 2:
tensor([[ 3., 13., 23.],
        [ 4., 14., 24.],
        [ 5., 15., 25.]])
Batch 3:
tensor([[ 6., 16., 26.],
        [ 7., 17., 27.],
        [ 8., 18., 28.]])
Batch 4:
tensor([[ 9., 19., 29.]])


In [7]:
from torch.utils.data import Dataset,Subset, TensorDataset, DataLoader, RandomSampler, BatchSampler,SequentialSampler
import torch
import random
import numpy
# Generate a list with 20 torch rand array with 3 columns(feature)
# The number of rows of each array is uncertain with a random number between 4 and 20
# To identify the array set the value of first row with indicies
DataSets = []
for i in range(20):
    length = random.randint(4, 10)
    dt = torch.randn(length,3)
    dt[0, ] = i
    DataSets.append((dt))
    
# create sub cloass of dataset
class Loader(Dataset):
    def __init__(self, dataset:list):
        self._datalist = dataset
    def __getitem__(self, idx):
        return self._datalist[idx]
    def __len__(self):
        return len(self._datalist)

# Creat hook function to pad the data sets in the batch 
# The goal is make all dataset in the batch having same length
# The padded datasets have to be packed 
def collate_fn_pad(batch):
    lengths = torch.tensor([t.shape[0] for t in batch])
    feature_batch = [t for t in batch]
    feature_batch = torch.nn.utils.rnn.pad_sequence(feature_batch, batch_first=True)    
    print(feature_batch)
    feature_batch = torch.nn.utils.rnn.pack_padded_sequence(
        feature_batch, lengths, batch_first=True, enforce_sorted=False
    )
    # Be aware print of the pack padded sequence doesn't show the real sequence of the data sets
    #
    print(feature_batch)
    return feature_batch

# Instance a Loader class with generated DataSets
LitDataModule = Loader(DataSets)

# Emulate datasets spliting
# Generate a sub datasets with index mod 4 unequal to 0, except the first
train_idx = numpy.arange(len(LitDataModule._datalist))
take_it = numpy.where(((train_idx)%4 != 0) | (train_idx == 0) )[0]
# pick datasets
newindex = list(set(train_idx) &set(take_it))
print(newindex)
# added a new attribute with the sub datasets
setattr(LitDataModule,'_traindata',Subset(LitDataModule,newindex))

# sort the datasets randomly
# sp is only the index of the datasets.
sp = RandomSampler(LitDataModule._traindata)

# bath the random sorted datasets
bs = BatchSampler(sampler= sp, batch_size = 5, drop_last=False) # the batch_size hier how many dataset are bathced to one

#create a dataloader for using, e.g. training
dtld = DataLoader(LitDataModule._traindata, collate_fn=collate_fn_pad, batch_sampler = bs, shuffle=False)

print('The collate_fn hook is not called\n')
for batch in dtld:
    print('\nThe collate_fn hook is called, dataset is padded and packed\n')
    paddeddt, lengths = torch.nn.utils.rnn.pad_packed_sequence(batch, batch_first=True)
    # Get the shortest datasets
    # To show changing of padding 0
    index_shortest = numpy.argmin(lengths)
    print('\n padded dataset')
    print(paddeddt[index_shortest])
    print('\n unpadded dataset')
    data = torch.nn.utils.rnn.unpad_sequence(paddeddt, lengths, batch_first=True)
    print(data[index_shortest])   
    
    break


[0, 1, 2, 3, 5, 6, 7, 9, 10, 11, 13, 14, 15, 17, 18, 19]
The collate_fn hook is not called

tensor([[[17.0000, 17.0000, 17.0000],
         [-0.7842,  0.9639,  0.9809],
         [-0.5302,  0.5440,  0.4207],
         [-0.3567,  0.1130, -1.4794],
         [ 0.5416, -0.1917,  2.5390],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000]],

        [[ 0.0000,  0.0000,  0.0000],
         [-0.2705,  1.3512, -0.9055],
         [-0.5812, -0.3083, -0.3738],
         [ 1.1661,  0.4092,  1.4787],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000]],

        [[18.0000, 18.0000, 18.0000],
         [-1.8059,  0.7799, -1.1656],
         [-1.1488,  1.0974,  1.6424],
         [ 0.2138, -1.0395,  1.1826],
         [ 1.3464,  0.2596,  0.2662],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0